In [3]:
import pandas as pd
import numpy as np

# --- Lire le fichier Excel (.xls) ---

df = pd.read_csv("data2.csv")

# --- Paramètre de la moyenne glissante ---
window = 10

# --- Colonnes numériques uniquement (on calcule mean/median/mode dessus) ---
num_cols = df.select_dtypes(include="number").columns.tolist()

# --- Fonctions de lissage ---
for col in num_cols:
    # Moyenne glissante
    df[f"{col}_ma{window}"] = df[col].rolling(window=window, min_periods=1).mean()
    # Médiane glissante
    df[f"{col}_median{window}"] = df[col].rolling(window=window, min_periods=1).median()
    # Mode glissant (valeur la plus fréquente sur la fenêtre)
    df[f"{col}_mode{window}"] = df[col].rolling(window=window, min_periods=1).apply(
        lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan,
        raw=False
    )

from IPython.display import display
display(df)

# --- Sauvegarder le fichier final ---
output_path = "data2_with_ma_median_mode.csv"
df.to_csv(output_path, index=False)

output_path


,age,gender,bmi,smoking_status,alcohol_consumption,exercise_level,diet_type,sun_exposure,income_level,latitude_region,...,has_numbness_tingling_mode10,has_memory_problems_ma10,has_memory_problems_median10,has_memory_problems_mode10,has_pale_skin_ma10,has_pale_skin_median10,has_pale_skin_mode10,has_multiple_deficiencies_ma10,has_multiple_deficiencies_median10,has_multiple_deficiencies_mode10
0,79,Male,24.8,Former,NaN,Active,Vegetarian,High,High,Mid,...,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,77,Female,39.9,Former,Moderate,Light,Omnivore,Low,Low,Low,...,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,24,Male,26.4,Former,Heavy,Moderate,Omnivore,Low,High,High,...,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,69,Male,23.1,Never,Heavy,Moderate,Vegetarian,High,Low,Low,...,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,63,Male,29.6,Never,NaN,Moderate,Vegetarian,Moderate,High,Low,...,0.0,0.20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,25,Female,21.6,Former,Heavy,Active,Pescatarian,Moderate,Middle,High,...,0.0,0.20,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0
3996,50,Male,29.5,Former,Moderate,Sedentary,Vegetarian,Low,High,Low,...,0.0,0.20,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0
3997,34,Female,24.8,Never,NaN,Active,Pescatarian,Moderate,Low,Low,...,0.0,0.30,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0
3998,39,Female,26.9,Former,Heavy,Active,Omnivore,High,Low,High,...,0.0,0.30,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0


'data2_with_ma_median_mode.csv'

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Lire ton fichier
file_path = "data2_with_ma_median_mode.csv"  # mets le bon chemin si besoin
df = pd.read_csv(file_path)

# 2) Features (comme sur ta capture)
features = [
    "vitamin_a_percent_rda",
    "hemoglobin_g_dl",
    "vitamin_c_percent_rda",
    "vitamin_e_percent_rda",
    "vitamin_b12_percent_rda",
    "vitamin_d_percent_rda",
    "folate_percent_rda",
    "calcium_percent_rda",
    "iron_percent_rda",
    "serum_vitamin_d_ng_ml",
    "serum_vitamin_b12_pg_ml",
    "serum_folate_ng_ml",
    "has_muscle_weakness",
    "has_night_blindness",
    "has_bleeding_gums",
    "has_fatigue",
    "has_bone_pain",
    "has_numbness_tingling",
    "has_memory_problems",
    "has_pale_skin",
    "diet_type"
]

target = "disease_diagnosis"

# 3) Vérifier que les colonnes existent
missing = [c for c in features + [target] if c not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes dans ton CSV : {missing}")

# 4) X et y
X = df[features]
y = df[target]

# 5) Colonnes numériques / catégorielles
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# 6) Pré-traitement (simple)
# - Numériques : remplacer NaN par médiane
# - Catégorielles : remplacer NaN par valeur la plus fréquente + OneHot
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

# 7) Modèle Random Forest (simple)
model = RandomForestClassifier(n_estimators=200, random_state=42)

# 8) Pipeline complet
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# 9) Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42, stratify=y
)

# 10) Entraîner + prédire
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# 11) Résultats
print("Accuracy =", accuracy_score(y_test, y_pred))
print("\nRapport détaillé :\n", classification_report(y_test, y_pred))
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred))


Accuracy = 0.925

Rapport détaillé :
                       precision    recall  f1-score   support

              Anemia       0.91      0.97      0.94        62
             Healthy       0.99      0.92      0.95        76
     Night_Blindness       1.00      1.00      1.00         6
Rickets_Osteomalacia       0.85      0.86      0.85        51
              Scurvy       1.00      1.00      1.00         5

            accuracy                           0.93       200
           macro avg       0.95      0.95      0.95       200
        weighted avg       0.93      0.93      0.93       200


Matrice de confusion :
 [[60  0  0  2  0]
 [ 0 70  0  6  0]
 [ 0  0  6  0  0]
 [ 6  1  0 44  0]
 [ 0  0  0  0  5]]
